# Import Necessary LIbraries

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import kurtosis, skew, entropy

# Feature Extraction From Raw Signal

In [2]:
def extract_features_from_raw(signal):
    """
    Translates a raw vibration signal into the 17-feature vector 
    required by the model.
    """
    # Time Domain Features
    f_mean = np.mean(signal)
    f_std = np.std(signal)
    f_rms = np.sqrt(np.mean(signal**2))
    f_max = np.max(signal)
    f_min = np.min(signal)
    f_p2p = np.ptp(signal)
    f_skew = skew(signal)
    f_kurt = kurtosis(signal)
    
    # Advanced Statistical Factors
    f_crest = f_max / f_rms if f_rms != 0 else 0
    f_shape = f_rms / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    f_impulse = f_max / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    f_clearance = f_max / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    
    # Entropy (Using Shannon entropy on signal distribution)
    hist, _ = np.histogram(signal, bins=100, density=True)
    f_entropy = entropy(hist + 1e-9) 
    
    # Frequency Domain Features (FFT)
    fft_vals = np.abs(np.fft.rfft(signal))
    f_fft_mean = np.mean(fft_vals)
    f_fft_std = np.std(fft_vals)
    f_fft_max = np.max(fft_vals)
    
    # Energy
    f_energy = np.sum(signal**2)
    
    
    feature_names = [
        'mean', 'std', 'rms', 'max', 'min', 'peak_to_peak', 'skewness', 
        'kurtosis', 'crest_factor', 'shape_factor', 'impulse_factor', 
        'clearance_factor', 'entropy', 'fft_mean', 'fft_std', 'fft_max', 'energy'
    ]
    
    features_df = pd.DataFrame([[
        f_mean, f_std, f_rms, f_max, f_min, f_p2p, f_skew, f_kurt,
        f_crest, f_shape, f_impulse, f_clearance, f_entropy,
        f_fft_mean, f_fft_std, f_fft_max, f_energy
    ]], columns=feature_names)
    
    return features_df

# Outer Race

In [3]:
df = pd.read_csv('set2&3_features.csv')

limit = 2000 
df['RUL_Clipped'] = df['RUL(min)'].apply(lambda x: x if x <= limit else limit)

# Features and New Target
X = df.drop(columns=['RUL(min)', 'RUL_Clipped'])
y = df['RUL_Clipped']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Model R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Average Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} minutes")

Model R2 Score: 0.9666
Average Error (MAE): 7.18 minutes


In [4]:
# 7. Save the model and scaler for the Inference Script
joblib.dump(model, 'outer_race_model.pkl')
joblib.dump(scaler, 'outer_race_scaler.pkl')
print("Outer race model and scaler saved successfully.")

Outer race model and scaler saved successfully.


In [5]:
# Load the saved model and scaler
outer_model = joblib.load('outer_race_model.pkl')
outer_scaler = joblib.load('outer_race_scaler.pkl')

In [6]:
def predict_rul_outer_race(raw_signal):
    # 1. Extract
    features = extract_features_from_raw(raw_signal)
    
    # 2. Scale
    features_scaled = outer_scaler.transform(features)
    
    # 3. Predict
    prediction = outer_model.predict(features_scaled)
    
    return max(0, prediction[0]) # RUL cannot be negative

# --- EXAMPLE USAGE ---
my_signal = np.random.normal(0, 0.1, 20480) 
estimated_min = predict_rul_outer_race(my_signal)
print(f"Remaining Life: {estimated_min:.2f} minutes")

Remaining Life: 983.05 minutes


---

# Inner Race

In [7]:
df = pd.read_csv('set1_inner_race_features.csv')

limit = 2000 
df['RUL_Clipped'] = df['RUL(min)'].apply(lambda x: x if x <= limit else limit)

# Features and New Target
X = df.drop(columns=['RUL(min)', 'RUL_Clipped'])
y = df['RUL_Clipped']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Model R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Average Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} minutes")

Model R2 Score: 0.7631
Average Error (MAE): 51.27 minutes


In [8]:
# 7. Save the model and scaler for the Inference Script
joblib.dump(model, 'inner_race_model.pkl')
joblib.dump(scaler, 'inner_race_scaler.pkl')
print("Inner race model and scaler saved successfully.")

Inner race model and scaler saved successfully.


In [9]:
# Load the saved model and scaler
inner_model = joblib.load('inner_race_model.pkl')
inner_scaler = joblib.load('inner_race_scaler.pkl')

In [10]:
def predict_rul_inner_race(raw_signal):
    # 1. Extract
    features = extract_features_from_raw(raw_signal)
    
    # 2. Scale
    features_scaled = inner_scaler.transform(features)
    
    # 3. Predict
    prediction = inner_model.predict(features_scaled)
    
    return max(0, prediction[0]) # RUL cannot be negative

# --- EXAMPLE USAGE ---
my_signal = np.random.normal(0, 0.1, 20480) 
estimated_min = predict_rul_inner_race(my_signal)
print(f"Remaining Life: {estimated_min:.2f} minutes")

Remaining Life: 583.65 minutes


---

# Ball

In [11]:
df = pd.read_csv('set1_ball_features.csv')
limit = 2000 
df['RUL_Clipped'] = df['RUL(min)'].apply(lambda x: x if x <= limit else limit)

# Features and New Target
X = df.drop(columns=['RUL(min)', 'RUL_Clipped'])
y = df['RUL_Clipped']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(X_train_scaled, y_train)

# 6. Performance Check
y_pred = model.predict(X_test_scaled)
print(f"Model R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Average Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} minutes")

Model R2 Score: 0.7854
Average Error (MAE): 54.89 minutes


In [12]:
# 7. Save the model and scaler for the Inference Script
joblib.dump(model, 'ball_model.pkl')
joblib.dump(scaler, 'ball_scaler.pkl')
print("Ball model and scaler saved successfully.")

Ball model and scaler saved successfully.


In [13]:
# Load the saved model and scaler
ball_model = joblib.load('ball_model.pkl')
ball_scaler = joblib.load('ball_scaler.pkl')

In [14]:
def predict_rul_ball(raw_signal):
    # 1. Extract
    features = extract_features_from_raw(raw_signal)
    
    # 2. Scale
    features_scaled = ball_scaler.transform(features)
    
    # 3. Predict
    prediction = ball_model.predict(features_scaled)
    
    return max(0, prediction[0]) # RUL cannot be negative

# --- EXAMPLE USAGE ---
my_signal = np.random.normal(0, 0.1, 20480) 
estimated_min = predict_rul_ball(my_signal)
print(f"Remaining Life: {estimated_min:.2f} minutes")

Remaining Life: 832.60 minutes


---

In [15]:
def load_signal_from_txt(txt_path):
    with open(txt_path, 'r', encoding='utf-8-sig') as file:
        text = file.read()

    values = []
    for line in text.splitlines():
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        cleaned_line = cleaned_line.replace(',', ' ')
        for token in cleaned_line.split():
            if token:
                values.append(float(token))

    if not values:
        raise ValueError(f'No numeric samples found in {txt_path}')

    return np.asarray(values, dtype=float)


def resolve_signal_input(signal_or_path):
    if isinstance(signal_or_path, str) and signal_or_path.lower().endswith('.txt'):
        return load_signal_from_txt(signal_or_path)

    signal = np.asarray(signal_or_path).squeeze()
    if signal.ndim != 1:
        signal = signal.reshape(-1)

    return signal

# Final RUL Estimator Fuction

In [16]:
def estimate_rul(fault_type, signal_or_path):
    signal = resolve_signal_input(signal_or_path)

    if fault_type == "Healthy":
        return "System Healthy", "RUL will be given by the designer"
    
    if fault_type == "Outer Race":
        prediction = predict_rul_outer_race(signal)
        if prediction == 2000:
            return "Actual RUL >= 2000 (Bearing in healthy period)", "Confidence=High (R2=0.97,MAE=7.18 minutes)"
        else:
            return round(float(prediction), 1), "Confidence=High (R2=0.97,MAE=7.18 minutes)"

    
    elif fault_type == "Inner Race":
        prediction = predict_rul_inner_race(signal)
        if prediction == 2000:
            return "Actual RUL >= 2000 (Bearing in healthy period)", "Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)"
        else:
            return round(float(prediction), 1), "Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)"

    
    elif fault_type == "ball":
        prediction = predict_rul_ball(signal)
        if prediction == 2000:
            return "Actual RUL >= 2000 (Bearing in healthy period)", "Confidence=Medium-High (R2=0.79,MAE=54.89 minutes)"
        else:
            return round(float(prediction), 1) ,"Confidence=Medium-High (R2=0.79,MAE=54.89 minutes)"
   
    else:
        return "Unknown Fault"

# Checking the function

In [17]:
test_path = 'test_signal/2003.10.22.12.06.24.txt'
estimate_rul('Inner Race', test_path)

(1992.5, 'Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)')

In [18]:
test_path = 'test_signal/2003.10.22.12.06.22.txt'
estimate_rul('Inner Race', test_path)

('Actual RUL >= 2000 (Bearing in healthy period)',
 'Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)')

In [19]:
test_path = 'test_signal/new_signal_values.txt'
estimate_rul('Inner Race', test_path)

(1992.5, 'Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)')

In [20]:
test_path = 'test_signal/2003.10.31.05.49.46.txt'
estimate_rul('Inner Race', test_path)

(1999.3, 'Confidence=Medium-High(R2=0.76,MAE=51.27 minutes)')

In [21]:
rul_details = estimate_rul('ball', test_path)